# Process and Job Control

## Quick notes

### What this covers
- What a process is, and how it differs from a program on disk
- `$$`, `$!`, `$PPID`, background jobs with `&`, and `wait`
- Job control (`jobs`, `set -m`) and why scripts mostly do not have it
- Signals: `kill`, `kill -0`, `trap`, cleanup on exit
- Exit statuses `126`, `127`, `128+n`, `130`
- `ps`, `pgrep`, `pkill`, zombies, process groups, `ulimit`
- Timeouts, parallel work, lock files, avoiding orphan processes

### Key points
- A **program** is a file. A **process** is a running copy of it with its own PID.
- `cmd &` starts a child and moves on. `$!` is its PID. `wait PID` blocks and gives you its exit status.
- `kill PID` sends SIGTERM. `kill -0 PID` sends nothing; it only asks "does this PID exist and may I signal it?".
- `trap 'code' EXIT` is the standard cleanup hook. Add `INT TERM` when you need to exit with a meaningful status.
- An exit status above 128 means "killed by signal (status - 128)".
- Scripts run with job control off. `%1`, `fg`, `bg` are for interactive shells.

### Gotchas (the part worth revising twice)
- `$$` inside `( ... )` is still the **parent's** PID. Do not `kill $$` in a subshell and expect the subshell to die.
- `wait` with no arguments always returns 0. To get each job's status, `wait` for each PID.
- A pipe-reading `$(...)` waits for background children that still hold its stdout open.
- A trap does not run while a foreground command is still running; it runs when that command ends (but it interrupts `wait` at once).
- `mkdir` is the portable atomic lock; "test then touch" has a race.
- macOS has no `flock`. Some recent macOS versions have `/usr/bin/timeout`, older ones do not; check with `command -v`.

### When it matters
- Cleaning up temp files and child processes when a script is interrupted
- Running several jobs in parallel and checking that every one succeeded
- Preventing two copies of a cron job from running at once
- Giving a stuck command a time limit

### One-line recap example
```bash
sleep 1 & pid=$!; wait "$pid"; echo "child exited with $?"
```
Files and the filesystem live in `03-Files-And-System/01-File-System-Ops`; error-handling options (`set -e`, `trap ERR`) are covered in `04-Scripting-Craft/01-Advanced-Bash`.

## 01. Programs vs processes

- A program is a file on disk (for example `/bin/sleep`). A process is one running instance of it.
- Every process has a PID (process id) and a PPID (its parent's PID). The kernel assigns them; they change on every run.
- Run the same program twice and you get two processes with two different PIDs.
- PIDs vary from run to run, so these demos only print true/false facts about them.

In [1]:
# setup: a %%bash cell magic that keeps ONE bash session alive between cells
import atexit, os, shutil, subprocess, tempfile, uuid
from IPython.core.magic import register_cell_magic

_WORK = tempfile.mkdtemp(prefix="bashnb_")
atexit.register(shutil.rmtree, _WORK, ignore_errors=True)
_ENV = {k: v for k, v in os.environ.items() if k not in ("CLICOLOR", "CLICOLOR_FORCE", "FORCE_COLOR", "PYDEVD_DISABLE_FILE_VALIDATION")}
_ENV.update(LC_ALL="C", TZ="UTC", HOME=_WORK, PS1="", PS4="+ ")
_session = None

def _start():
    global _session
    _session = subprocess.Popen(["bash", "--noprofile", "--norc"], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=_WORK, env=_ENV)

def _clean(text):
    return text.replace(os.path.join(_WORK, "script"), "script").replace(_WORK, "<work>")

def _run_persistent(code, timeout=60):
    global _session
    if _session is None or _session.poll() is not None:
        _start()
    token = uuid.uuid4().hex
    path = os.path.join(_WORK, "script")
    with open(path, "w") as fh:
        fh.write(code + "\n")
    _session.stdin.write(". '%s' </dev/null; printf '\\n%s %%d\\n' $?\n" % (path, token))
    _session.stdin.flush()
    out, status = [], None
    import threading
    result = {}
    def reader():
        while True:
            line = _session.stdout.readline()
            if not line:
                result["status"] = None; return
            if line.startswith(token + " "):
                result["status"] = int(line.split()[1]); return
            out.append(line)
    t = threading.Thread(target=reader, daemon=True); t.start(); t.join(timeout)
    os.remove(path)
    if t.is_alive():
        _session.kill(); _session = None
        return "".join(out) + "[timed out]\n", 124
    if result.get("status") is None:       # the shell itself exited (e.g. the cell ran `exit N`)
        code = _session.wait(); _session = None
        return "".join(out) + "(the bash session ended; the next cell starts a fresh one)\n", code
    text = "".join(out)
    if text.endswith("\n\n"):
        text = text[:-1]          # drop the blank line our status marker adds
    return text, result.get("status") if result.get("status") is not None else 1

def _run_fresh(code, timeout=60):
    path = os.path.join(_WORK, "script")
    with open(path, "w") as fh:
        fh.write(code + "\n")
    done = subprocess.run(["bash", "--noprofile", "--norc", path], stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, cwd=_WORK, env=_ENV, timeout=timeout)
    os.remove(path)
    return done.stdout, done.returncode

@register_cell_magic
def bash(line, cell):
    """Run the cell in bash. Flags: fresh (new shell), expect_fail (show non-zero status instead of raising)."""
    flags = line.split()
    text, status = (_run_fresh if "fresh" in flags else _run_persistent)(cell)
    print(_clean(text), end="" if text.endswith("\n") or not text else "\n")
    if status != 0:
        if "expect_fail" in flags:
            print("[exit status %d]" % status)
        else:
            raise RuntimeError("bash cell exited with status %d" % status)

In [2]:
%%bash
case $$ in *[!0-9]*|'') echo "pid is a number: no";; *) echo "pid is a number: yes";; esac
case $PPID in *[!0-9]*|'') echo "ppid is a number: no";; *) echo "ppid is a number: yes";; esac

# a child's parent is us
sh -c 'echo $PPID' > ppid.txt
[ "$(cat ppid.txt)" = "$$" ] && echo "child's PPID equals our \$\$: yes"
rm -f ppid.txt

# the same program run twice -> two processes -> two PIDs
a=$(sh -c 'echo $$'); b=$(sh -c 'echo $$')
[ "$a" != "$b" ] && echo "two runs of sh got different PIDs: yes"

# what is this process running?
echo "our program: $(ps -o comm= -p $$ | sed 's|.*/||')"

pid is a number: yes
ppid is a number: yes
child's PPID equals our $$: yes
two runs of sh got different PIDs: yes
our program: bash


## 02. `$$` does not change inside `( )`

- `$$` always means the PID of the **original** shell, even inside a subshell `( ... )` or `$( ... )`.
- A subshell is a forked copy of the shell, so it has its own real PID, but `$$` is not updated.
- `$BASHPID` gives the real PID of the current subshell, but it only exists in bash 4+. On bash 3.2 compare via a child process instead.
- Use `$$` for unique temp file names in the main script; in subshells it can collide, so create the name before forking.

In [3]:
%%bash
main=$$
sub=$( echo $$ )
[ "$main" = "$sub" ] && echo "\$\$ inside \$( ) equals the parent's \$\$: yes"

# a child of the subshell sees the subshell's real PID as its PPID
sub_real=$( sh -c 'echo $PPID' )
[ "$sub_real" != "$main" ] && echo "the subshell really is a different process: yes"

echo "BASHPID set here: ${BASHPID:-no (bash 3.2 has no BASHPID)}"

$$ inside $( ) equals the parent's $$: yes
the subshell really is a different process: yes
BASHPID set here: no (bash 3.2 has no BASHPID)


## 03. Background jobs with `&`

- `cmd &` starts `cmd` in the background. The shell continues immediately.
- `$!` is the PID of the most recent background process. It is empty if you have not started one.
- Save `$!` into your own variable right away; the next `&` overwrites it.
- Background output still goes to your terminal (or the notebook), mixed with other output. Redirect it to a file for tidy results.

In [4]:
%%bash
echo "before any job, \$! is: [${!:-}]"

( sleep 0.2; echo "background job finished" > bg.out ) &
first=$!
sleep 0.05 &
second=$!
[ "$first" != "$second" ] && echo "each & gets its own PID: yes"
kill -0 "$first" 2>/dev/null && echo "job 1 is still running right now: yes"

wait "$first" "$second"
cat bg.out
rm -f bg.out

before any job, $! is: []
each & gets its own PID: yes
job 1 is still running right now: yes
background job finished


## 04. `wait`: block, and collect the exit status

- `wait PID` blocks until that child ends and returns **its exit status** in `$?`.
- `wait` with no arguments waits for **all** children and always returns 0, so it hides failures.
- `wait` on a PID that is not your child returns 127. (Bash 3.2 also remembers the status of the most recently reaped child, so waiting twice on the same PID repeats the old status. Newer bash versions usually answer 127 here, so do not rely on it.)
- A signal-killed child gives `128 + signal number` (see section 12).

In [5]:
%%bash
( exit 7 ) &
pid=$!
wait "$pid"
echo "wait PID returned the child's status: $?"

( exit 3 ) & ( exit 4 ) &
wait
echo "bare wait returned: $? (always 0, the failures are hidden)"

( exit 9 ) & pid=$!
wait "$pid"; echo "first wait: $?"
wait "$pid" 2>/dev/null; echo "second wait on the same pid: $? (bash remembers the last reaped status)"
wait 99999999 2>/dev/null; echo "wait on a pid that is not our child: $?"

wait PID returned the child's status: 7
bare wait returned: 0 (always 0, the failures are hidden)
first wait: 9
second wait on the same pid: 9 (bash remembers the last reaped status)
wait on a pid that is not our child: 127


## 05. Jobs in a script: `jobs` and job control

- Interactive shells have **job control** (the `m` flag): each job gets its own process group, and `fg`, `bg`, `%1` and Ctrl-Z work.
- A script runs with job control **off**. `jobs` still lists background children, but there are no process groups and no `fg`/`bg`.
- Turn it on with `set -m`. This is rarely a good idea in scripts, but you need it for section 21 (killing a whole group).
- Check which flags are on with `echo $-`; an `m` means job control is active.

In [6]:
%%bash
case $- in *m*) echo "job control (m) is on";; *) echo "job control (m) is off";; esac
sleep 5 &
echo "jobs in this shell: $(jobs | wc -l | tr -d ' ')"
kill $! ; wait $! 2>/dev/null
echo "jobs after kill + wait: $(jobs | wc -l | tr -d ' ')"

job control (m) is off
jobs in this shell: 1
jobs after kill + wait: 0


In [7]:
%%bash fresh
set -m
case $- in *m*) echo "job control (m) is on";; *) echo "job control (m) is off";; esac
sleep 5 &
pid=$!
# with job control each job is its own process group leader
[ "$(ps -o pgid= -p $pid | tr -d ' ')" = "$pid" ] && echo "background job leads its own process group: yes"
jobs | sed 's/^\[[0-9]*\][+-]* *//'
kill $pid; wait $pid 2>/dev/null
echo "done"

job control (m) is on
background job leads its own process group: yes
Running                 sleep 5 &
done


## 06. Signals: names, numbers, and `kill`

- A signal is a small message sent to a process. The process can handle it, ignore it, or die from it.
- `kill PID` sends SIGTERM (15, "please stop"). `kill -9 PID` sends SIGKILL (cannot be caught or ignored), which gives the process no chance to clean up. Prefer TERM first.
- Common ones: HUP 1 (terminal closed / reload), INT 2 (Ctrl-C), QUIT 3, KILL 9, TERM 15, USR1/USR2 (user defined), CHLD (child ended).
- `kill -l` lists them. Given a number it prints the name; given a name it prints the number.
- Signal numbers differ between systems for the less common signals (USR1, STOP, CHLD). Use names in scripts.

In [8]:
%%bash
echo "names for 1 2 3 9 15:"; kill -l 1 2 3 9 15
echo "numbers for HUP INT TERM KILL:"; kill -l HUP INT TERM KILL

names for 1 2 3 9 15:
HUP
INT
QUIT
KILL
TERM
numbers for HUP INT TERM KILL:
1
2
15
9


## 07. `kill -0`: is this process alive?

- `kill -0 PID` sends no signal. It only reports through its exit status whether the PID exists and you may signal it.
- Exit 0 means yes. Non-zero means it is gone (or belongs to someone else).
- Use it to wait for something to start or stop, or to check a PID file.
- Gotcha: PIDs are reused. A stale PID file may point at a different, unrelated process.

In [9]:
%%bash
sleep 5 &
pid=$!
kill -0 "$pid" 2>/dev/null && echo "alive before kill: yes"
kill "$pid"
wait "$pid" 2>/dev/null
echo "wait status after TERM: $? (128 + 15)"
kill -0 "$pid" 2>/dev/null || echo "alive after kill: no"

# polling loop: wait up to ~1s for a process to disappear
sleep 0.3 & pid=$!
n=0
while kill -0 "$pid" 2>/dev/null && [ "$n" -lt 20 ]; do sleep 0.1; n=$((n+1)); done
kill -0 "$pid" 2>/dev/null || echo "process went away by itself"
wait "$pid" 2>/dev/null

alive before kill: yes
wait status after TERM: 143 (128 + 15)
alive after kill: no
process went away by itself


## 08. `trap`: handling signals in fresh shells

- `trap 'commands' SIGNAL...` runs `commands` when the shell receives that signal.
- `kill -TERM $$` makes a script signal itself, which is the easiest way to demo traps without a real Ctrl-C.
- The cells below use `fresh` shells so a signal can never hurt the notebook session.
- A trap handler runs, then the script **continues** after the interrupted command, unless the handler calls `exit`.

In [10]:
%%bash fresh
trap 'echo "caught INT"'  INT
trap 'echo "caught TERM"' TERM
trap 'echo "caught HUP"'  HUP
kill -INT  $$
kill -TERM $$
kill -HUP  $$
echo "script kept running after three signals"

caught INT
caught TERM
caught HUP
script kept running after three signals


In [11]:
%%bash fresh expect_fail
# handler that exits: the usual shape for INT/TERM
trap 'echo "got TERM, exiting"; exit 143' TERM
trap 'echo "EXIT trap runs too"' EXIT
kill -TERM $$
echo "never printed"

got TERM, exiting
EXIT trap runs too
[exit status 143]


## 09. When does a trap run?

- Bash only runs a trap **between** commands. While a foreground command such as `sleep 1` is running, the trap waits until it finishes.
- The builtin `wait` is different: a trapped signal interrupts it immediately, and `wait` returns `128 + signal`.
- So, for long waits, background the work and use `wait`; the trap then fires promptly.

In [12]:
%%bash fresh
trap 'echo "trap ran"' TERM
( sleep 0.2; kill -TERM $$ ) &
sleep 1
echo "foreground sleep finished first, then the trap ran (see order above)"

trap ran
foreground sleep finished first, then the trap ran (see order above)


In [13]:
%%bash fresh
trap 'echo "trap ran at once"' TERM
sleep 5 & worker=$!
( sleep 0.2; kill -TERM $$ ) &
wait "$worker"
echo "wait was interrupted, status: $? (128 + 15)"
kill "$worker"; wait "$worker" 2>/dev/null
true

trap ran at once
wait was interrupted, status: 143 (128 + 15)


## 10. Cleanup with `trap ... EXIT`

- `EXIT` is a pseudo-signal: it fires when the shell exits for any reason: normal end, `exit`, an error under `set -e`, and also when a script is killed by SIGINT, SIGTERM or SIGHUP (bash runs the trap, then dies from the signal).
- Put `rm -rf "$tmp"` there, so temp files are removed on every path out.
- Create the temp dir **before** installing the trap, or make the cleanup tolerate an empty variable.
- The EXIT trap does not change the exit status unless it calls `exit`.
- Gotcha: `SIGKILL` (9) cannot be trapped, so the EXIT trap does not run; plan for stale temp files in that case.

In [14]:
%%bash fresh
tmp=$(mktemp -d)
trap 'rm -rf "$tmp"; echo "cleanup done"' EXIT
echo data > "$tmp/work.txt"
[ -f "$tmp/work.txt" ] && echo "work file exists while running"
echo "$tmp" > saved_tmp_path
exit 0

work file exists while running
cleanup done


In [15]:
%%bash
[ -d "$(cat saved_tmp_path)" ] && echo "temp dir still exists: yes" || echo "temp dir still exists: no (trap removed it)"
rm -f saved_tmp_path

temp dir still exists: no (trap removed it)


In [16]:
%%bash
# the trap also runs when the script is killed by TERM, and the parent still sees 143
( bash -c 'tmp=$(mktemp -d); trap "rm -rf \"$tmp\"; echo cleanup on TERM" EXIT; kill -TERM $$; echo never printed'
  echo "status seen by the parent shell: $?" ) 2>&1 | sed 's/^.*Terminated.*$/(shell reports: Terminated)/'

cleanup on TERM
(shell reports: Terminated)
status seen by the parent shell: 143


## 11. Trap bookkeeping: replace, list, reset, ignore

- A second `trap` for the same signal **replaces** the first; it does not add to it. To chain handlers, put both in one function.
- `trap -p` prints the active traps in re-usable form; `trap` with no arguments does the same.
- `trap - SIGNAL` resets the signal to its default behaviour.
- `trap '' SIGNAL` **ignores** the signal, and children started afterwards inherit the ignore.
- A caught trap is not passed on to other programs, and a `( )` subshell does not run it either. On bash 3.2 `trap -p` prints nothing inside `( )`; newer bash may still list the parent's traps there. Ignored signals (`trap '' SIG`) stay ignored in both.

In [17]:
%%bash fresh
trap 'echo first' USR1
trap 'echo second' USR1      # replaces the first
kill -USR1 $$
echo "--- trap -p shows:"
trap -p USR1
trap - USR1                  # back to default
echo "--- after reset trap -p prints: [$(trap -p USR1)]"
trap '' USR1                 # ignore
kill -USR1 $$
echo "ignored USR1 and survived"
# an ignored signal stays ignored in children
sh -c 'kill -USR1 $$; echo "child also ignored USR1"'

second
--- trap -p shows:
trap -- 'echo second' SIGUSR1
--- after reset trap -p prints: []
ignored USR1 and survived
child also ignored USR1


In [18]:
%%bash fresh expect_fail
# one cleanup function used by several signals
cleanup() { echo "cleanup(): $1"; }
trap 'cleanup INT;  exit 130' INT
trap 'cleanup TERM; exit 143' TERM
trap 'cleanup EXIT' EXIT
kill -INT $$

cleanup(): INT
cleanup(): EXIT
[exit status 130]


## 12. Exit statuses that mean something

- `0` success; `1`-`125` are the program's own error codes.
- `126`: the file exists but cannot be executed (for example no execute permission, or it is a directory).
- `127`: command not found.
- `128 + n`: killed by signal `n`. So 130 = Ctrl-C (INT, 2), 137 = KILL (9), 143 = TERM (15), 129 = HUP (1).
- `$?` holds the status of the last command only. For every stage of a pipeline use the `PIPESTATUS` array, or `set -o pipefail` (see `02-Streams-And-Text/01-Pipes-And-Redirection` and `04-Scripting-Craft/01-Advanced-Bash`).
- When the shell reports a signal death on stderr it prints a line containing the PID; the cells below hide it with `2>/dev/null`.

In [19]:
%%bash
printf 'echo not allowed\n' > noexec.sh        # no execute bit
./noexec.sh; echo "status of non-executable file: $?"
bash -c 'no_such_command_xyz' ; echo "status of unknown command: $?"
( sh -c 'kill -TERM $$'; echo "TERM-killed child: $?" ) 2>/dev/null
( sh -c 'kill -KILL $$'; echo "KILL-killed child: $?" ) 2>/dev/null
( sh -c 'kill -INT  $$'; echo "INT-killed child:  $?" ) 2>/dev/null
( sh -c 'kill -HUP  $$'; echo "HUP-killed child:  $?" ) 2>/dev/null
rm -f noexec.sh

script: line 2: ./noexec.sh: Permission denied
status of non-executable file: 126
bash: no_such_command_xyz: command not found
status of unknown command: 127
TERM-killed child: 143
KILL-killed child: 137
INT-killed child:  130
HUP-killed child:  129


## 13. "Command not found" in detail

- The shell searches each directory in `$PATH`. No match gives status 127 and a message on stderr.
- A name with a slash is never searched: `./tool` goes straight to that file. A missing file there is 127, a present-but-not-executable file is 126.
- Check before you call: `command -v tool` prints the path (or nothing) and returns non-zero if missing. Prefer it over `which`, whose behaviour differs between systems.
- The shell remembers where it found commands (`hash`). If a program moves, `hash -r` forgets the old place.
- Bash 4+ has `command_not_found_handle` (not available in 3.2).

In [20]:
%%bash expect_fail
if command -v definitely_missing_tool >/dev/null 2>&1; then echo "found"; else echo "definitely_missing_tool: not installed"; fi
echo "sh is at a path: $(command -v sh | grep -c '^/')"

bash -c './no_such_file' ; echo "slash + missing file: $?"
bash -c 'x=1 no_such_command_xyz' ; echo "assignment prefix does not help: $?"
PATH= /bin/bash -c 'ls' 2>&1 | sed 's|^/bin/bash: ||'; echo "empty PATH makes even ls unfindable"

definitely_missing_tool: not installed
sh is at a path: 1
bash: ./no_such_file: No such file or directory
slash + missing file: 127
bash: no_such_command_xyz: command not found
assignment prefix does not help: 127
ls: No such file or directory
empty PATH makes even ls unfindable


## 14. Replacing the shell with `exec`

- `exec cmd` **replaces** the current shell with `cmd`. There is no return; lines after it never run.
- Use it as the last step of a wrapper script so no extra shell process lingers and signals reach `cmd` directly.
- `exec` with only redirections (`exec 3>file`, `exec >log 2>&1`) changes the shell's own file descriptors and does not replace it.
- In `( exec cmd )` only the subshell is replaced, which is a cheap way to avoid one extra fork.
- If `exec` cannot find the command, a non-interactive shell exits with 127.

In [21]:
%%bash fresh
echo "before exec"
exec echo "this process is now echo"
echo "never printed"

before exec
this process is now echo


In [22]:
%%bash
# exec with only redirections keeps the shell
( exec 3> fd3.txt; echo "written through fd 3" >&3; exec 3>&-; echo "subshell continues after exec redirect" )
cat fd3.txt; rm -f fd3.txt

# subshell vs exec: ( ) forks a copy; exec swaps the program
( cd / ; echo "subshell sees cwd: $(pwd)" )
echo "parent cwd unchanged: $([ "$PWD" != / ] && echo yes)"

subshell continues after exec redirect
written through fd 3
subshell sees cwd: /
parent cwd unchanged: yes


## 15. Subshells and background work in command substitution

- `$(...)` and `( ... )` run in a forked copy. Variable changes inside do not come back.
- Gotcha: `$( cmd & )` does not return until the background child closes its inherited stdout, so a `&` in a command substitution can still block.
- Fix: redirect the background command's output (`> /dev/null 2>&1`) so it no longer holds the pipe.

In [23]:
%%bash
# slow: command substitution waits for the pipe to close
SECONDS=0
x=$( sleep 2 & echo hi )
echo "got: $x"
[ "$SECONDS" -ge 2 ] && echo "it waited for the background sleep: yes"

# fast: background child no longer holds the pipe
SECONDS=0
x=$( sleep 2 > /dev/null 2>&1 & echo hi )
echo "got: $x"
[ "$SECONDS" -lt 2 ] && echo "returned immediately: yes"

got: hi
it waited for the background sleep: yes
got: hi
returned immediately: yes


## 16. `ps`, `pgrep`, `pkill`

- `ps -p PID -o fields=` shows chosen columns for one process; a trailing `=` removes the header. `-o comm=` is the command name, `-o pgid=` the process group, `-o stat=` the state.
- `pgrep -P PARENT` lists children of a PID; `pgrep -x name` matches the exact name.
- `pkill` is `pgrep` plus a signal. Always narrow it (`-P $$`, `-x`, `-f` carefully) or you may kill other people's processes.
- Syntax is portable for these flags, but the **columns** differ: macOS `ps aux` and GNU `ps aux` look similar, `ps -o` field names differ for a few (use `comm`, `pid`, `ppid`, `pgid`, `stat`).
- Do not parse `ps` output for anything but quick checks; keep PIDs from `$!`.

In [24]:
%%bash
# the shell prints a 'Terminated' notice (with a PID) for killed jobs; send stderr away for a moment
exec 3>&2 2>/dev/null
sleep 30 & p0=$!
sleep 30 & p1=$!
echo "command name of a child: $(ps -o comm= -p $p1 | sed 's|.*/||')"
echo "its parent is us: $([ "$(ps -o ppid= -p $p1 | tr -d ' ')" = "$$" ] && echo yes || echo no)"
echo "our sleep children: $(pgrep -P $$ -x sleep | wc -l | tr -d ' ')"

pkill -P $$ -x sleep
wait $p0 $p1 2>/dev/null
exec 2>&3 3>&-
echo "our sleep children after pkill: $(pgrep -P $$ -x sleep | wc -l | tr -d ' ')"

command name of a child: sleep
its parent is us: yes
our sleep children: 2
our sleep children after pkill: 0


## 17. Zombies (defunct processes)

- When a child ends, the kernel keeps a tiny entry until the parent calls `wait` to read its exit status. That entry is a **zombie** (state `Z`).
- Bash reaps its background children for you, so you rarely see zombies from plain `&` jobs.
- A zombie appears when the parent is alive but never waits (for example it `exec`'d into a program that does not care about children).
- You cannot `kill` a zombie; it is already dead. End its parent, and the system adopts and reaps it.
- So: if you background children in a long-running script, `wait` for them eventually.

In [25]:
%%bash
# parent 'sh' starts a short child, then exec's into a long sleep that never waits for it
( sh -c 'sleep 0.1 & echo $! > child.pid; exec sleep 3' ) &
parent=$!
sleep 0.6
child=$(cat child.pid)
echo "child state letter: $(ps -o stat= -p $child | cut -c1)   (Z = zombie)"
kill "$parent"; wait "$parent" 2>/dev/null
sleep 0.3
kill -0 "$child" 2>/dev/null && echo "zombie still listed: yes" || echo "zombie gone after its parent died: yes"
rm -f child.pid

child state letter: Z   (Z = zombie)
zombie gone after its parent died: yes


## 18. Timeouts: with and without `timeout`

- Some recent macOS versions ship `/usr/bin/timeout`. Older macOS and most BSDs do not. Linux has GNU `timeout`.
- `timeout 1 cmd` exits 124 if it had to stop `cmd`. Check `command -v timeout` first.
- The portable fallback: run the command in the background and start a watcher that sleeps then kills it.
- The watcher must be stopped when the command finishes early, and its own `sleep` must be killed too, otherwise it lingers.

In [26]:
%%bash
if command -v timeout >/dev/null 2>&1; then
    timeout 1 sleep 5; echo "timeout status: $? (124 = timed out)"
else
    echo "no timeout command here"
fi

timeout status: 124 (124 = timed out)


In [27]:
%%bash
# portable fallback: returns 124 on timeout, otherwise the command's own status
run_with_timeout() {
    local secs=$1; shift
    local flag="to.flag.$$"
    rm -f "$flag"
    "$@" & local pid=$!
    ( sleep "$secs" & local sp=$!
      trap 'kill "$sp" 2>/dev/null; exit 0' TERM
      wait "$sp" && { : > "$flag"; kill "$pid" 2>/dev/null; } ) 2>/dev/null &
    local watcher=$!
    wait "$pid" 2>/dev/null; local status=$?
    kill "$watcher" 2>/dev/null; wait "$watcher" 2>/dev/null
    if [ -f "$flag" ]; then rm -f "$flag"; return 124; fi
    return "$status"
}

run_with_timeout 1 sleep 5;  echo "slow command: $? (124 = timed out)"
run_with_timeout 5 sh -c 'exit 3'; echo "fast failing command: $? (its own status)"
run_with_timeout 5 true;     echo "fast ok command: $?"
echo "stray sleeps left over: $(pgrep -P $$ -x sleep | wc -l | tr -d ' ')"

slow command: 124 (124 = timed out)
fast failing command: 3 (its own status)
fast ok command: 0
stray sleeps left over: 0


## 19. Running work in parallel: `&` + `wait`

- Start each job with `&`, remember each PID, then `wait` for each PID to get its status.
- Bash 3.2 arrays work for this: `pids=( ... )`, `pids+=($!)` (the `+=` form exists in 3.1+).
- Do not print from jobs directly if order matters; write to per-job files and read them in order.
- A bare `wait` only tells you the jobs finished, not whether they succeeded.

In [28]:
%%bash
mkdir -p par && cd par
pids=()
for i in 1 2 3 4; do
    ( sleep "0.$((5 - i))"; echo "result $i" > "out.$i"; [ "$i" -ne 3 ] ) &
    pids+=($!)
done

failed=0
for idx in 0 1 2 3; do
    if wait "${pids[$idx]}"; then :; else echo "job $((idx + 1)) failed"; failed=$((failed + 1)); fi
done
echo "failed jobs: $failed"
cat out.1 out.2 out.3 out.4
cd .. && rm -rf par

job 3 failed
failed jobs: 1
result 1
result 2
result 3
result 4


## 20. Parallel with `xargs -P`

- `xargs -P N` runs up to N commands at once. BSD and GNU xargs both support `-P`.
- `-n 1` gives one argument per command; `-I {}` substitutes it. Output of parallel jobs interleaves, so `sort` it or write to separate files.
- `-0` with `find -print0` handles odd file names. If a job fails, GNU `xargs` exits 123 while BSD `xargs` (macOS) exits 1, so only test for "non-zero".
- Unlike a handwritten loop, `xargs` does not start more than N at a time, which is the main point.

In [29]:
%%bash
printf '%s\n' 1 2 3 4 5 6 | xargs -P 3 -n 1 sh -c 'echo "task $0 ok"' | sort

echo "--- failure propagation:"
printf '%s\n' 1 2 3 | xargs -n 1 -P 2 sh -c '[ "$0" != 2 ]' ; echo "xargs status after a failed job: $? (BSD value; GNU gives 123)"

task 1 ok
task 2 ok
task 3 ok
task 4 ok
task 5 ok
task 6 ok
--- failure propagation:
xargs status after a failed job: 1 (BSD value; GNU gives 123)


## 21. Process groups and `kill -- -PGID`

- Each process belongs to a **process group**. Normally all programs in a pipeline or job share one.
- `kill -SIG -- -PGID` (negative number) signals the **whole group**, which also ends children and grandchildren that `kill PID` would miss.
- Ctrl-C in a terminal goes to the foreground group; that is how it stops a whole pipeline.
- In a script, groups only exist per-job with `set -m` (job control). Without it, children share the script's group (and the script is usually not its leader, so `kill -- -$$` fails or hits the wrong group). Never signal a group that contains your own script.
- `--` stops `kill` reading the negative number as an option.

In [30]:
%%bash fresh
set -m
( sleep 30 & sleep 30 & wait ) &
leader=$!
sleep 0.3
echo "processes in the job's group: $(ps -ax -o pid=,pgid= | awk -v g="$leader" '$2==g' | wc -l | tr -d ' ')"
kill -TERM -- -"$leader"
wait "$leader" 2>/dev/null
sleep 0.2
echo "group members left after kill -- -PGID: $(ps -ax -o pgid= | awk -v g="$leader" '$1==g' | wc -l | tr -d ' ')"

processes in the job's group: 3
group members left after kill -- -PGID: 0


## 22. Locks with `mkdir` (no `flock` on macOS)

- `mkdir` either creates the directory or fails, in one atomic step. That makes it a safe lock; `[ -f lock ] || touch lock` has a gap where two copies both pass.
- Always release the lock in an `EXIT` trap, and keep the PID inside so a stale lock can be detected with `kill -0`.
- Linux has `flock(1)`; macOS does not. Do not write scripts that need it if they must run on macOS.
- A lock directory can still go stale if the process is killed with SIGKILL; the PID check handles that. Taking over a stale lock has its own small race (two copies can both steal it), and a PID can be reused, so treat this as good enough for cron-style jobs, not for strict exclusion.

In [31]:
%%bash
command -v flock >/dev/null 2>&1 && echo "flock exists here" || echo "flock: not available here"

acquire_lock() {
    local lock=$1
    if mkdir "$lock" 2>/dev/null; then echo $$ > "$lock/pid"; return 0; fi
    local owner; owner=$(cat "$lock/pid" 2>/dev/null)
    if [ -n "$owner" ] && ! kill -0 "$owner" 2>/dev/null; then   # owner is dead: steal it
        rm -rf "$lock"; mkdir "$lock" 2>/dev/null && { echo $$ > "$lock/pid"; return 0; }
    fi
    return 1
}
release_lock() { rm -rf "$1"; }

acquire_lock job.lock && echo "first acquire: ok"
acquire_lock job.lock || echo "second acquire: busy (we hold it)"
release_lock job.lock
acquire_lock job.lock && echo "after release: ok again"
release_lock job.lock

# stale lock from a dead process is taken over
mkdir job.lock; echo 99999999 > job.lock/pid
acquire_lock job.lock && echo "stale lock taken over: yes"
release_lock job.lock

flock: not available here
first acquire: ok
second acquire: busy (we hold it)
after release: ok again
stale lock taken over: yes


## 23. Avoiding orphans

- If a script exits while its background children are still running, they keep going (they are "orphans", re-parented to PID 1).
- Keep the PIDs and kill them in an `EXIT` trap so Ctrl-C or an error does not leave stray processes.
- `jobs -p` lists the PIDs of the shell's background jobs, even without job control; it is a handy way to kill "everything I started".
- `kill` of a subshell does not kill its children. Use `exec` in the subshell, or a process group (section 21).

In [32]:
%%bash fresh
trap 'kill $(jobs -p) 2>/dev/null' EXIT
sleep 30 > /dev/null 2>&1 &
echo $! > worker.pid
echo "script exits now; trap kills the worker"

script exits now; trap kills the worker


In [33]:
%%bash
sleep 0.3
kill -0 "$(cat worker.pid)" 2>/dev/null && echo "worker survived: yes (leak!)" || echo "worker survived: no (trap cleaned it up)"
rm -f worker.pid

worker survived: no (trap cleaned it up)


In [34]:
%%bash fresh
# without a trap the child is orphaned
sleep 30 > /dev/null 2>&1 &
echo $! > worker.pid
echo "script exits without cleanup"

script exits without cleanup


In [35]:
%%bash
kill -0 "$(cat worker.pid)" 2>/dev/null && echo "orphan still running: yes"
kill "$(cat worker.pid)" 2>/dev/null; sleep 0.2
kill -0 "$(cat worker.pid)" 2>/dev/null || echo "orphan killed manually"
rm -f worker.pid

orphan still running: yes
orphan killed manually


## 24. `nohup` and `disown`

- When a terminal closes, the shell sends **SIGHUP** to its jobs and they die.
- `nohup cmd` starts `cmd` with SIGHUP ignored (and output sent to `nohup.out` if stdout is a terminal). It survives logout.
- `disown` is the bash builtin that removes a job from the shell's job table, so the shell will not HUP it at exit.
- Modern alternatives are `tmux`/`screen`, `systemd` services or `launchd` jobs. `nohup` is the quick tool.
- A process that ignores HUP via `nohup` still dies on TERM and KILL.

In [36]:
%%bash
( sh -c 'kill -HUP $$; echo "survived HUP"'; echo "plain sh status: $?" ) 2>/dev/null
nohup sh -c 'kill -HUP $$; echo "survived HUP"' > nohup.log 2>&1
echo "under nohup status: $?"; cat nohup.log; rm -f nohup.log

sleep 5 & pid=$!
disown "$pid"
echo "jobs listed after disown: $(jobs | wc -l | tr -d ' ')"
kill "$pid"

plain sh status: 129
under nohup status: 0
survived HUP
jobs listed after disown: 0


## 25. `ulimit`: per-process limits

- `ulimit -a` shows resource limits (open files, CPU seconds, processes, stack, core size...). Defaults differ a lot between systems, so scripts should not print them.
- Each limit has a **soft** value (enforced) and a **hard** ceiling. `-S` and `-H` choose which one to show or set; a normal user can lower the hard limit but not raise it again.
- Lowering a limit inside `( ... )` affects only that subshell.
- Useful ones: `-n` open files, `-t` CPU seconds, `-u` max processes, `-c` core dump size.

In [37]:
%%bash
echo "limits reported (labels only):"
ulimit -a | sed 's/ *\(([^)]*)\).*/ \1/' | sort | head -8
echo "..."
( ulimit -n 64; echo "open files inside subshell: $(ulimit -n)" )
echo "soft limit <= hard limit here: $( [ "$(ulimit -Sn)" = unlimited ] || [ "$(ulimit -Hn)" = unlimited ] || [ "$(ulimit -Sn)" -le "$(ulimit -Hn)" ] && echo yes || echo no )"

limits reported (labels only):
core file size (blocks, -c)
cpu time (seconds, -t)
data seg size (kbytes, -d)
file size (blocks, -f)
max locked memory (kbytes, -l)
max memory size (kbytes, -m)
max user processes (-u)
open files (-n)
...
open files inside subshell: 64
soft limit <= hard limit here: yes


## 26. Bash 4+ items (not run here)

These need bash 4 or newer; `/bin/bash` 3.2 on macOS cannot run them.

**`wait -n`** (bash 4.3): return as soon as any one background job ends, which makes a rolling worker pool easy.
```bash
# needs bash 4.3+ (not run here)
for f in *.dat; do
    while (( $(jobs -r | wc -l) >= 4 )); do wait -n; done
    process "$f" &
done
wait
```
On 3.2, use `xargs -P` (section 20) or poll `kill -0`.

**`coproc`** (bash 4): a background process with two pipes you can read and write.
```bash
# needs bash 4+ (not run here)
coproc UPPER { while read -r l; do echo "${l^^}"; done; }
echo hello >&"${UPPER[1]}"
read -r reply <&"${UPPER[0]}"     # reply is now HELLO
kill "$UPPER_PID"
```
On 3.2, use named pipes (`mkfifo`) instead.

**`$BASHPID`**, **`${var,,}`**-style helpers and `command_not_found_handle` are also 4+.

## Recap
- Know the three IDs: `$$` (me), `$!` (last background child), `$PPID` (my parent).
- `wait PID` is the only way to get a child's exit status; a bare `wait` returns 0.
- Clean up with `trap ... EXIT`; signals from outside normally go through the same cleanup.
- `128 + n` means killed by signal n. 126 = cannot execute, 127 = not found.
- `mkdir` for locks; `command -v timeout` or the background+kill pattern for time limits.